##Step 2: Data Collection

###import necessary libraries

In [ ]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer
import warnings
warnings.filterwarnings('ignore')

###Load Dataset

In [ ]:
ratings = pd.read_csv('u.data',sep='\t',names=['user_id','item_id','rating','timestamp'])
ratings
movies = pd.read_csv('u.item',sep = '|',encoding='latin-1',header=None)
movies = movies[[0,1]]
movies.columns = ['item_id','title']
movies_data = pd.merge(ratings,movies,on='item_id')
movies_data['timestamp'] = pd.to_datetime(movies_data['timestamp'],unit='s')
movies_data

,user_id,item_id,rating,timestamp,title
0,196,242,3,1997-12-04 15:55:49,Kolya (1996)
1,186,302,3,1998-04-04 19:22:22,L.A. Confidential (1997)
2,22,377,1,1997-11-07 07:18:36,Heavyweights (1994)
3,244,51,2,1997-11-27 05:02:03,Legends of the Fall (1994)
4,166,346,1,1998-02-02 05:33:16,Jackie Brown (1997)
...,...,...,...,...,...
99995,880,476,3,1997-11-22 05:10:44,"First Wives Club, The (1996)"
99996,716,204,5,1997-11-17 19:39:03,Back to the Future (1985)
99997,276,1090,1,1997-09-20 22:49:55,Sliver (1993)
99998,13,225,2,1997-12-17 22:52:36,101 Dalmatians (1996)


##Step 3: Data Understanding

###Perform Initial Investigation

In [ ]:
movies_data.shape

(100000, 5)

In [ ]:
movies_data.isna().sum()

,0
user_id,0
item_id,0
rating,0
timestamp,0
title,0


In [ ]:
movies_data.describe()

,user_id,item_id,rating,timestamp
count,100000.00000,100000.000000,100000.000000,100000
mean,462.48475,425.530130,3.529860,1997-12-31 00:40:51.488619904
min,1.00000,1.000000,1.000000,1997-09-20 03:05:10
25%,254.00000,175.000000,3.000000,1997-11-13 19:18:29.500000
50%,447.00000,322.000000,4.000000,1997-12-22 21:42:24
75%,682.00000,631.000000,4.000000,1998-02-23 18:53:04
max,943.00000,1682.000000,5.000000,1998-04-22 23:10:38
std,266.61442,330.798356,1.125674,NaN


In [ ]:
movies_data['title'].nunique()

1664

In [ ]:
movies_data['user_id'].nunique()

943

###Building All Recommendation Engine From Popularity to Hybrid

####1.Popularity Based

In [ ]:
def popularity_recommendation(top_n=5):
  pop = movies_data.groupby('title')['rating'].agg(['mean','count'])
  pop.columns = ['avg_rating','count']
  pop = pop[pop['count'] > 50 ]
  pop = pop.sort_values(by='avg_rating',ascending=False)
  return pop.head(top_n).index.tolist()


####2. Trend Based

In [ ]:
def trending_recommendation(top_n=5):
  recent = movies_data[movies_data['timestamp'] > '1998-01-01']
  trend = recent.groupby('title')['rating'].agg(['mean','count'])
  trend.columns = ['avg_rating','count']
  trend = trend.sort_values(by='count',ascending=False)
  return trend.head(top_n).index.tolist()


####3.Content Based

In [ ]:
cv = CountVectorizer(stop_words='english')
matrix = cv.fit_transform(movies['title'])
content_similarity = cosine_similarity(matrix)



In [ ]:
def content_recommendation(movie_name,top_n=5):
  if movie_name not in movies['title'].values:
    print('Movie Not Found')
  idx = movies[movies['title'] == movie_name].index[0]
  scores = list(enumerate(content_similarity[idx]))
  scores = sorted(scores,key=lambda x: x[1],reverse=True)[1:top_n+1]
  return [movies.iloc[i[0]].title for i in scores]



####4.Item Based Collarborative

In [ ]:
collab_pvt = movies_data.pivot_table(index='user_id',columns='title',values='rating').fillna(0)
collab_similarity = cosine_similarity(collab_pvt.T)
collab_df = pd.DataFrame(collab_similarity,index=collab_pvt.columns,columns=collab_pvt.columns)


In [ ]:
def collaborative_recommendation(movie_name,top_n=5):
  if movie_name not in collab_df.columns:
    print('Movie Not Found')
  scores = collab_df[movie_name].sort_values(ascending=False)[1:top_n+1]
  return scores.index.tolist()



####5.Hybrid Recommendation

In [ ]:
def hybrid_recommendation(movie_name):
  content = content_recommendation(movie_name)
  collab = collaborative_recommendation(movie_name)
  combined = list(dict.fromkeys(content + collab))
  return combined[:5]


In [ ]:
def recommend_all(movie_name):
  print(f'Recommendation for: {movie_name}')
  print('-------------')
  print('\Popularity:')
  print(popularity_recommendation())
  print('\nTrending:')
  print(trending_recommendation())
  print('\nContent-based:')
  print(content_recommendation(movie_name))
  print('\nCollaborative Based:')
  print(collaborative_recommendation(movie_name))
  print('\nhybrid:')
  print(hybrid_recommendation(movie_name))


In [ ]:
recommend_all('Star Wars (1977)')

Recommendation for: Star Wars (1977)
-------------
\Popularity:
['Close Shave, A (1995)', "Schindler's List (1993)", 'Wrong Trousers, The (1993)', 'Casablanca (1942)', 'Wallace & Gromit: The Best of Aardman Animation (1996)']

Trending:
['Titanic (1997)', 'Contact (1997)', 'Star Wars (1977)', 'Air Force One (1997)', 'Scream (1996)']

Content-based:
['Lone Star (1996)', 'Annie Hall (1977)', 'Audrey Rose (1977)', 'Evening Star, The (1996)', "Pete's Dragon (1977)"]

Collaborative Based:
['Return of the Jedi (1983)', 'Raiders of the Lost Ark (1981)', 'Empire Strikes Back, The (1980)', 'Toy Story (1995)', 'Godfather, The (1972)']

hybrid:
['Lone Star (1996)', 'Annie Hall (1977)', 'Audrey Rose (1977)', 'Evening Star, The (1996)', "Pete's Dragon (1977)"]
